In [15]:
!wget -O sparql-anything.jar https://github.com/SPARQL-Anything/sparql.anything/releases/download/v1.1.0/sparql-anything-v1.1.0.jar
!java -version

--2026-01-22 10:43:27--  https://github.com/SPARQL-Anything/sparql.anything/releases/download/v1.1.0/sparql-anything-v1.1.0.jar
Resolving github.com (github.com)... 140.82.113.4
Connecting to github.com (github.com)|140.82.113.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/303967701/d4b57144-968f-4781-a18e-ead6b6eb0da1?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-01-22T11%3A33%3A14Z&rscd=attachment%3B+filename%3Dsparql-anything-v1.1.0.jar&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-01-22T10%3A32%3A25Z&ske=2026-01-22T11%3A33%3A14Z&sks=b&skv=2018-11-09&sig=kjJr%2FZywjZOv5QLIiq1mecptYB8ImosIkK4SAnfvyVo%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc2OTA4MjIwNywibmJmIjoxNzY5MDc4NjA3LCJwYXRoIj

In [ ]:
import subprocess
import os

jar_path = "sparql-anything.jar"
input_csv = "Dati_Gare_Invernali.csv"
output_file = "dati_importati.ttl"
query_file = "query_temp.sparql"


sparql_query = f"""
PREFIX xyz: 
PREFIX xsd: 
PREFIX sio: 
PREFIX rdfs: 

CONSTRUCT {{
    ?garaURI a sio:GaraOlimpica ;
             rdfs:label ?eventLabel ;
             sio:haData ?dateStr ;  # Importiamo la data come stringa per ora
             sio:haPartecipante ?atletaURI .

    ?atletaURI a sio:AtletaOlimpico ;
               sio:haNome ?athleteName ;
               sio:haEta ?ageTyped ;
               sio:praticaSport ?sportURI ;
               sio:vinceMedaglia ?medalURI .
}}
WHERE {{
    # NOTA: Qui specifichiamo csv.delimiter=;
    SERVICE  {{
        ?row xyz:EventID ?eventID ;
             xyz:Date ?dateStr ;
             xyz:Athlete ?athleteName ;
             xyz:Age ?ageStr ;
             xyz:Sport ?sportStr ;
             xyz:Medal ?medalStr .
    }}

    # Creazione URI
    BIND(URI(CONCAT(STR(sio:), ?eventID)) AS ?garaURI)
    BIND(STR(?eventID) AS ?eventLabel)
    BIND(URI(CONCAT(STR(sio:), REPLACE(?athleteName, " ", ""))) AS ?atletaURI)

    # Casting Età
    BIND(xsd:integer(?ageStr) AS ?ageTyped)

    # Mapping Sport
    BIND(
        IF(?sportStr = "AlpineSki", sio:SciAlpinoSport,
            IF(?sportStr = "Biathlon", sio:BiathlonSport,
                IF(?sportStr = "Snowboard", sio:SnowboardSport, sio:SportInvernale)
            )
        ) AS ?sportURI
    )

    # Mapping Medaglia
    BIND(
        IF(?medalStr = "Gold", sio:Oro,
            IF(?medalStr = "Silver", sio:Argento, sio:Bronzo)
        ) AS ?medalURI
    )
}}
"""


with open(query_file, "w") as f:
    f.write(sparql_query)

print("Esecuzione ...")
cmd = ["java", "-jar", jar_path, "-q", query_file, "-o", output_file]

try:
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode == 0:
        print(f" File creato: {output_file}")
        if os.path.exists(output_file):
            with open(output_file, "r") as f:
                print(f.read()[:500])
    else:
        print(" Error:")
        print(result.stderr)
except Exception as e:
    print(e)

Esecuzione con separatore punto e virgola...
 File creato: dati_importati.ttl
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX sio:  <http://www.semanticweb.org/ModSem/SportInvernaliOlimpici/>
PREFIX xsd:  <http://www.w3.org/2001/XMLSchema#>
PREFIX xyz:  <http://sparql.xyz/facade-x/data/>

sio:JohannesKlaebo  a      sio:AtletaOlimpico;
        sio:haEta          29;
        sio:haNome         "Johannes Klaebo";
        sio:praticaSport   sio:SportInvernale;
        sio:vinceMedaglia  sio:Argento .

sio:CrossCountry15km  a     sio:GaraOlimpica;
        rdfs:labe
